# 第10課 - 生產環境中的 AI 代理

在本課中，您將學習使用 Microsoft Agent Framework（Python）的 AI 代理 <strong>生產模式</strong>。內容涵蓋：

- <strong>可觀察性</strong> — 為代理互動添加時間記錄與日誌
- <strong>評估</strong> — 使用評估代理來打分回應質量
- <strong>成本管理</strong> — 代幣優化與模型選擇策略

場景是一個 <strong>旅遊代理</strong>，協助用戶規劃行程，並在上層加設監控與評估。


## 設定


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## 產品考慮事項

將 AI 代理從原型轉移到生產環境需要仔細注意三大支柱：

1. <strong>可觀察性</strong> — 您需要了解代理在做什麼、花了多長時間以及調用了哪些工具。沒有追蹤和日誌記錄，調試生產問題幾乎是不可能的。

2. <strong>評估</strong> — 自動化品質檢查確保代理的回應隨著時間保持準確、完整和有用。評估代理可以根據定義的標準評分回應。

3. <strong>成本管理</strong> — 令牌使用量直接影響成本。諸如提示優化、模型選擇和快取等策略有助於在不犧牲質量的情況下控制開支。


## 建立一個可觀察的代理

我們定義旅行工具並用計時來包裝代理調用，以便我們能監測延遲。在生產環境中，你會整合 OpenTelemetry 或類似的追蹤後端。


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## 成本管理策略

控制成本對於生產 AI 代理至關重要。以下是關鍵策略：

| 策略 | 描述 |
|---|---|
| <strong>提示優化</strong> | 保持系統指令簡潔。移除冗餘上下文以減少輸入標記數。 |
    "| <strong>模型選擇</strong> | 對於分類或提取等簡單任務，使用較小、較便宜的模型（例如 GPT-5-mini），並將複雜推理任務保留給較大的模型。 |\n",
| <strong>快取</strong> | 快取工具結果和常見查詢，以避免重複的 API 呼叫。 |
| <strong>標記預算</strong> | 設定 `max_tokens` 限制以防止回應過長造成意外成本。 |
| <strong>批次處理</strong> | 盡可能將多個用戶查詢組合成單一 API 呼叫。 |

實務上，一種分層方法效果很好：將簡單請求導向快速且廉價的模型，僅對複雜查詢升級到更強大的模型。


## 摘要

在本課程中，你學會了如何：

1. <strong>新增可觀察性</strong> 至代理互動中，透過計時與日誌記錄，為追蹤與監控奠定基礎。
2. 使用評估代理自動<strong>評估代理回應</strong>，評分完整性、準確性與幫助性。
3. 透過提示優化、模型選擇、快取和標記預算來<strong>控制成本</strong>。

這些生產模式有助確保你的 AI 代理在大規模運行時可靠、可衡量且具成本效益。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
